#### Structured Output

Models can be required to provide their response in a format matching a given schema. THis is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.


#### Pydantic
- Pydantic models provide the richest feature set with field validation, descriptions and structures.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x0000029C1665FCB0>, default_metadata=(), model_kwargs={})

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description='Title of the movie.')
    year: int = Field(description='Year the movie was released.')
    director: str = Field(description='Director of the movie.')
    rating: float = Field(description='Rating of the movie out of 10.')
    

In [4]:
model_with_str = model.with_structured_output(Movie)
model_with_str

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x0000029C1665FCB0>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'properties': {'title': {

In [5]:
response = model_with_str.invoke("Provide details of movie Conjuring 2")
response

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Movie(title='The Conjuring 2', year=2016, director='James Wan', rating=7.3)

### Message output alongside parsed strcuture

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(...,description='Title of the movie.')
    year: int = Field(...,description='Year the movie was released.')
    director: str = Field(...,description='Director of the movie.')
    rating: float = Field(...,description='Rating of the movie out of 10.')
    
model_with_structure = model.with_structured_output(Movie, include_raw=True)

#### Nested Structure

In [9]:
from pydantic import BaseModel, Field

class Actors(BaseModel):
    name: str
    role: stra


class Movie(BaseModel):
    title: str = Field(...,description='Title of the movie.')
    year: int = Field(...,description='Year the movie was released.')
    cast: list[Actors]
    genre: list[str]
    director: str = Field(...,description='Director of the movie.')
    rating: float = Field(...,description='Rating of the movie out of 10.')
    
model_with_structure = model.with_structured_output(Movie, include_raw=True)
# model_with_structure
response = model_with_structure.invoke("Provide details of movie Conjuring 2")
response

{'raw': AIMessage(content='{"title": "The Conjuring 2", "year": 2016, "cast": [{"name": "Vera Farmiga", "role": "Lorraine Warren"}, {"name": "Patrick Wilson", "role": "Ed Warren"}, {"name": "Madison Wolfe", "role": "Janet Hodgson"}, {"name": "Frances O\'Connor", "role": "Peggy Hodgson"}, {"name": "Simon McBurney", "role": "Maurice Grosse"}, {"name": "Franka Potente", "role": "Anita Gregory"}], "genre": ["Horror", "Mystery", "Thriller"], "director": "James Wan", "rating": 7.3}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a15366-e8c9-72b2-bc1c-2f86dcd827d8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 367, 'total_tokens': 376, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 221}}),
 'parsed': Movie(title='The Conjuring 2', year=2016, cast=[Actors(name='Vera Farmiga', role='Lorraine W

#### TypeDict

- provides a simple alternative using python's built-in typing. Ideal when you don't need runtime validation

In [11]:
from typing_extensions import TypedDict, Annotated

class Actors(TypedDict):
    name:str
    role:str

class Movie(TypedDict):
    """A movie with details"""
    title: Annotated[str, 'Title of movie']
    year: Annotated[int, 'Year Released']
    cast: list[Actors]
    genre: list[str]
    director: Annotated[str, 'Director of the movie']
    rating: Annotated[int, 'Movie Rating out of 10']   

model_with_typeDict = model.with_structured_output(Movie)
response = model_with_typeDict.invoke('Provide details of The Nun')
response

{'title': 'The Nun',
 'year': 2018,
 'cast': [{'name': 'Taissa Farmiga', 'role': 'Sister Irene'},
  {'name': 'Demián Bichir', 'role': 'Father Burke'},
  {'name': 'Jonas Bloquet', 'role': 'Frenchie'}],
 'genre': ['Horror', 'Mystery', 'Thriller'],
 'director': 'Corin Hardy',
 'rating': 6}

In [13]:
model.profile

{'name': 'Gemini 2.5 Flash',
 'release_date': '2025-06-17',
 'last_updated': '2025-06-17',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

#### DataClasses

- A class typically containing mainly data, although there aren't really any restrictions.
- Can create using @dataclass

In [16]:
from pydantic import BaseModel, Field

class Actors(BaseModel):
    name: str
    role: str   


class Movie(BaseModel):
    title: str = Field(...,description='Title of the movie.')
    year: int = Field(...,description='Year the movie was released.')
    cast: list[Actors]
    genre: list[str]
    director: str = Field(...,description='Director of the movie.')
    rating: float = Field(...,description='Rating of the movie out of 10.')
    
model_with_structure = model.with_structured_output(Movie, include_raw=True)
# model_with_structure
response = model_with_structure.invoke("Provide details of movie Conjuring 2")

print(response['parsed'])

title='The Conjuring 2' year=2016 cast=[Actors(name='Vera Farmiga', role='Lorraine Warren'), Actors(name='Patrick Wilson', role='Ed Warren'), Actors(name='Madison Wolfe', role='Janet Hodgson'), Actors(name="Frances O'Connor", role='Peggy Hodgson'), Actors(name='Simon McBurney', role='Maurice Grosse')] genre=['Horror', 'Thriller', 'Mystery'] director='James Wan' rating=7.3


In [19]:
## using dataclass

from dataclasses import dataclass
from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information of a person"""
    name: str
    email: str
    phone: str

model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
llm = model.with_structured_output(ContactInfo)
# model = create_agent(
#     model='google:gemini-2.5-flash',
#     response_format=ContactInfo
# )

ValueError: Unsupported schema type <class 'type'>